# FCC-ee HEB V26.2 LCC impedance budget and 0.4 mm tracking wake

CERN's FCC Week 2026 hardware table specifies **10 extraction magnet modules per beam** and a **418 mm magnetic length** per module. The Point B LCC layout puts two independent extraction sections on the common Booster orbit. The full turn therefore includes **20 modules**. Each section contributes 4.18 m in the existing magnetic-length-scaled impedance model.

- [Extraction hardware, Ruf, FCC Week 2026, slide 10](https://indico.cern.ch/event/1552126/contributions/7132555/attachments/3292652/5887883/FCC-Week-2026_Kicker_Magnets_and_Generators_for_the_FCC-ee.pdf#page=10)
- [Point B LCC layout, Dutheil and Yue, slide 3](https://indico.cern.ch/event/1625611/contributions/6846330/attachments/3199104/5694904/2026_01_13_Booster_PB_GHC_vs_LCC.pdf#page=3)
- [CERN 2026 integration inventory, Booster extraction row](https://indico.cern.ch/event/1552126/contributions/7128305/attachments/3291969/5886498/FCC%20Week%20Poster%20-%20FCC-ee%20Integration%20Requirements%20for%20Injection-Extraction%20Equipment%202026.pdf)

The circular RW length subtracts the extraction length once, using the same accounting as V26.2.1. The transverse extraction term uses the ring-average beta unless measured or designed module positions are supplied in the native PyWIT route. The extraction impedance uses the length-scaled 20 m collider injection spectrum as a proxy. The two separate Point L damper striplines, their beta weights, longitudinal HOM-only RF, transverse IDDEFIX fits, and the Cu/150 nm NEG wall with 0.3 µm roughness follow the reference export.

Open this notebook under the repository's Example directory and choose **Run All**. The default calculation reads the published V26.2.1 component tables and recomputes all totals. With the same component shapes, grids and beta weights, impedance and Gaussian bunch wake are linear in the two changed multipliers. This route uses the already calculated 10 THz RW wake and the matched RF wakes at their original resolution. The native PyWIT route below rebuilds from the input spectra and the locally installed model.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import importlib
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.constants import c as speed_of_light

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'fcc_ee_booster_pywit_model' / 'data' / 'total').is_dir()
     and (path / 'Example').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the FCC-ee HEB repository root or its Example directory.')
DATA = PROJECT_ROOT / 'fcc_ee_booster_pywit_model' / 'data'

## 1. Configuration

Use two sections for the full LCC ring. One section reproduces the extraction and RW normalization of the published reference and can be used as a control. The native route requires the local editable PyWIT and HEB layout helpers, the original 20 m optimized kicker inputs, the stripline spectra, the matched RF inputs, and the IW2D wiggler cache or executable.

In [ ]:
RUN_MODE = 'published_components'  # or 'native_pywit'
N_EXTRACTION_SECTIONS = 2
MODULES_PER_EXTRACTION_SECTION = 10
MAGNETIC_LENGTH_PER_MODULE_M = 0.418
COLLIDER_EXTRACTION_REFERENCE_M = 20.0

# Optional: one position per module from the Point B equipment layout.
# The native route interpolates both beta functions at those positions.
EXTRACTION_MODULE_S_POSITIONS_M = None

N_HEB_DAMPERS = 2
N_COLLIDER_STRIPLINES = 12
ACTIVE_LENGTH_PER_DAMPER_M = 1.0
DAMPER_S_POSITIONS_M = [79_151.6315, 79_476.8315]

if N_EXTRACTION_SECTIONS not in (1, 2):
    raise ValueError('Select one extraction section or both LCC sections.')
if RUN_MODE not in ('published_components', 'native_pywit'):
    raise ValueError('Choose published_components or native_pywit.')
N_EXTRACTION_MODULES = N_EXTRACTION_SECTIONS * MODULES_PER_EXTRACTION_SECTION
L_EXTRACTION_M = N_EXTRACTION_MODULES * MAGNETIC_LENGTH_PER_MODULE_M
EXTRACTION_FACTOR = L_EXTRACTION_M / COLLIDER_EXTRACTION_REFERENCE_M


# Native HEB V26.2 settings, retained from the uploaded notebook
ENERGY_EV = 20e9
OPTICS_TFS = DATA/'optics'/'v26_2'/'heb_ring_z.tfs'
KICKER_ARCHIVE = None
WIGGLER_CACHE = None
FLATCHAMBER_EXE = None

# Numerical grids. The 10 MHz budget is for browsing/plots, not HOM loss factors.
F_BUDGET = np.linspace(0.0, 50e9, 5001)
WAKE_SIGMA_M = 0.4e-3
WAKE_FMAX_HZ = 10e12
WAKE_DF_HZ = 100e6
WAKE_WINDOW_MM = (-3.0, 20.0)

SAVE_PROJECT_DATA = True
SAVE_FIGURES = False
# Select the matched transverse RF impedance and wake from the fit notebook.
RF_TRANSVERSE_MODE = 'iddefix_fit'  # 'cst_native' or 'iddefix_fit'
transverse_tags = {'cst_native': 'RF_CSTnative', 'iddefix_fit': 'RF_IDDEFIXdipoles'}
if RF_TRANSVERSE_MODE not in transverse_tags:
    raise ValueError(f'Choose RF_TRANSVERSE_MODE from {list(transverse_tags)}')
CASE_TAG = (
    f'v26_2_HOMonly_Damper_N{N_HEB_DAMPERS}_{transverse_tags[RF_TRANSVERSE_MODE]}'
    f'_ExtK{N_EXTRACTION_MODULES}'
)
TOTAL_EXPORT_DIR = DATA/'total'/CASE_TAG
FIGURES_DIR = PROJECT_ROOT/'figures'/CASE_TAG

RF_IMP_FILENAME = 'Z_single_rf_cavity_long_IDDEFIX_HOM_only_ADAPTIVE_0to50GHz.txt'
RF_WAKE_FILENAME = 'W_single_rf_cavity_long_IDDEFIX_HOM_only_sigma0.4mm.txt'
RF_IMP_PATH = DATA/'rf_cavities'/RF_IMP_FILENAME
RF_WAKE_PATH = DATA/'rf_cavities'/RF_WAKE_FILENAME
RF_TRANSVERSE_MANIFEST_PATH = DATA/'rf_cavities'/'RF_IDDEFIX_transverse_budget_manifest.json'

DAMPER_FILENAMES = {
    'long':'Z_long_stripkickers26.txt',
    'x':'Z_dipx_stripkickers26.txt',
    'y':'Z_dipy_stripkickers26.txt',
}
DAMPER_WAKE_FILENAME = 'W_stripkickers26.txt'


REFERENCE_TAG = 'v26_2_HOMonly_Damper_N2_RF_IDDEFIXdipoles'
REFERENCE_DIR = DATA / 'total' / REFERENCE_TAG
REFERENCE_PREFIX = f'HEB_{REFERENCE_TAG}'
NATIVE_FAMILIES = ('RW', 'Wiggler', 'Kicker', 'RF', 'BPM', 'Bellows')
FAMILIES = NATIVE_FAMILIES + ('Damper',)
plane_stems = {'long': 'long', 'x': 'dipx', 'y': 'dipy'}

if N_HEB_DAMPERS < 1 or N_COLLIDER_STRIPLINES < 1:
    raise ValueError('Damper counts must be positive.')
if not OPTICS_TFS.is_file():
    raise FileNotFoundError(f'V26.2 Twiss table missing: {OPTICS_TFS}')
if EXTRACTION_MODULE_S_POSITIONS_M is not None:
    positions = np.asarray(EXTRACTION_MODULE_S_POSITIONS_M, dtype=float)
    if positions.shape != (N_EXTRACTION_MODULES,) or not np.all(np.isfinite(positions)):
        raise ValueError('Supply one finite s coordinate for each extraction module.')
    if len(np.unique(positions)) != N_EXTRACTION_MODULES:
        raise ValueError('Extraction modules need distinct positions.')
    if RUN_MODE == 'published_components':
        raise ValueError('Use native_pywit to apply local extraction beta weights.')

## 2. Read the V26.2.1 reference and check the pipe length

In [ ]:
reference_inputs = []


def reference_path(suffix):
    path = REFERENCE_DIR / f'{REFERENCE_PREFIX}_{suffix}'
    if not path.is_file():
        raise FileNotFoundError(f'Missing V26.2.1 reference export: {path}')
    reference_inputs.append(path)
    return path


def read_reference_csv(suffix):
    return pd.read_csv(reference_path(suffix), float_precision='round_trip')


reference_manifest = json.loads(reference_path('EXPORT_MANIFEST.json').read_text(encoding='utf-8'))
reference_inventory = read_reference_csv('inventory.csv').set_index('component')
reference_provenance = read_reference_csv('source_provenance.csv')
reference_damper_locations = read_reference_csv('Damper_locations.csv')

with OPTICS_TFS.open(encoding='utf-8') as optics:
    circumference_m = next(
        float(fields[3]) for line in optics
        if (fields := line.split())[:2] == ['@', 'LENGTH']
    )
bellows_m = float(reference_inventory.loc['Bellows', 'physical_length_m'])
wiggler_m = float(reference_inventory.loc['Wiggler', 'physical_length_m'])
reference_extraction_m = float(reference_inventory.loc['Kicker', 'physical_length_m'])
reference_rw_m = float(reference_inventory.loc['RW', 'physical_length_m'])
rw_length_m = circumference_m - bellows_m - wiggler_m - L_EXTRACTION_M
np.testing.assert_allclose(reference_rw_m, circumference_m - bellows_m - wiggler_m - reference_extraction_m,
                           rtol=0, atol=1e-6)
if rw_length_m <= 0:
    raise ValueError('The remaining circular pipe length must be positive.')
if EXTRACTION_MODULE_S_POSITIONS_M is not None and (
        positions.min() < 0 or positions.max() >= circumference_m):
    raise ValueError('Extraction module positions must lie in the V26.2 ring.')

rw_ratio = rw_length_m / reference_rw_m
extraction_ratio = L_EXTRACTION_M / reference_extraction_m
length_summary = pd.DataFrame({
    'Quantity': ['Extraction sections', 'Extraction modules', 'Extraction magnetic length [m]',
                 'Extraction longitudinal impedance scale', 'Circular RW length [m]'],
    'V26.2.1 reference': [1, int(reference_inventory.loc['Kicker', 'count']), reference_extraction_m,
                         reference_extraction_m / COLLIDER_EXTRACTION_REFERENCE_M, reference_rw_m],
    'This run': [N_EXTRACTION_SECTIONS, N_EXTRACTION_MODULES, L_EXTRACTION_M,
                 EXTRACTION_FACTOR, rw_length_m],
}).set_index('Quantity')
display(length_summary.round(9))
print(f'RW impedance and wake scale relative to reference = {rw_ratio:.12f}')
print('Extraction transverse weighting:',
      'ring-average beta (provisional)' if EXTRACTION_MODULE_S_POSITIONS_M is None else 'local module beta')

## 3. Native PyWIT source calculation

In [ ]:
if RUN_MODE == 'native_pywit':
    from pywit.component import Component
    from pywit.element import Element
    from pywit.model import Model
    from fcc_ee_booster_pywit_model import HEBModel
    from fcc_ee_booster_pywit_model.package_paths import base_dir

    if Path(base_dir).resolve() / 'data' != DATA:
        raise RuntimeError('Use the editable HEB package from the same project checkout as this notebook.')
    if not HEBModel.__module__.endswith('.heb_gold_tl'):
        raise RuntimeError('The V26.2 native HEBModel in heb_gold_tl.py is required.')
    for path in (RF_IMP_PATH, RF_WAKE_PATH, RF_TRANSVERSE_MANIFEST_PATH):
        if not path.is_file():
            raise FileNotFoundError(f'Missing matched IDDEFIX export: {path}')

    # Native source arrays stay untouched. Refuse ambiguous duplicate filenames.
    def find_unique_source(filename):
        candidates = [p for p in DATA.rglob(filename) if p.is_file()]
        if not candidates:
            raise FileNotFoundError(
                f'{filename} not found under {DATA}. Put the four stripkickers26 files inside data/ '
                'or any of its subfolders.'
            )
        if len(candidates) > 1:
            raise RuntimeError(f'Multiple copies of {filename}; choose one explicitly: {candidates}')
        return candidates[0]


    def read_complex_impedance_table(path):
        dat = np.loadtxt(path, comments='#', usecols=(0,1,2), ndmin=2)
        if len(dat) < 3 or not np.all(np.isfinite(dat)):
            raise ValueError(f'Bad/non-finite impedance source: {path}')
        freq_ghz, re, im = dat.T
        if np.any(np.diff(freq_ghz) <= 0) or freq_ghz[0] < 0:
            raise ValueError(f'Frequency axis must be strictly increasing GHz: {path}')
        if not 0 < freq_ghz[-1] <= 1e5:
            raise ValueError(f'Implausible GHz frequency range: {path}')
        return freq_ghz*1e9, re+1j*im

    DAMPER_PATHS = {pl: find_unique_source(fname) for pl,fname in DAMPER_FILENAMES.items()}
    STRIPLINE_DATA = {pl:read_complex_impedance_table(path) for pl,path in DAMPER_PATHS.items()}
    RF_FREQ, RF_Z_SINGLE = read_complex_impedance_table(RF_IMP_PATH)

    rf_transverse_manifest = json.loads(RF_TRANSVERSE_MANIFEST_PATH.read_text(encoding='utf-8'))
    if rf_transverse_manifest['schema_version'] != 1:
        raise ValueError(f'Unsupported RF manifest version in {RF_TRANSVERSE_MANIFEST_PATH}')
    if not np.isclose(WAKE_SIGMA_M, rf_transverse_manifest['sigma_z_m'], atol=1e-15, rtol=0):
        raise ValueError('WAKE_SIGMA_M must match the 0.4 mm RF exports.')


    def matching_rf_path(record):
        path = RF_TRANSVERSE_MANIFEST_PATH.parent / record['file']
        if sha256(path.read_bytes()).hexdigest() != record['sha256']:
            raise ValueError(f'{path.name} differs from the RF manifest. Rerun the RF fit notebook.')
        return path


    RF_IMP_PATHS = {'long': RF_IMP_PATH}
    RF_WAKE_PATHS = {'long': RF_WAKE_PATH}
    RF_TRANSVERSE_DATA = {}
    for plane in ('x', 'y'):
        records = rf_transverse_manifest['planes'][plane]
        pair = records[RF_TRANSVERSE_MODE]
        RF_IMP_PATHS[plane] = matching_rf_path(pair['impedance'])
        RF_WAKE_PATHS[plane] = matching_rf_path(pair['wake'])
        RF_TRANSVERSE_DATA[plane] = read_complex_impedance_table(RF_IMP_PATHS[plane])
        f, _ = RF_TRANSVERSE_DATA[plane]
        if f[0] != 0 or not np.isclose(f[-1], 50e9, atol=1, rtol=0):
            raise ValueError(f'Transverse RF source must span 0–50 GHz: {RF_IMP_PATHS[plane]}')
        print(f'RF dipole {plane} ({RF_TRANSVERSE_MODE}): {len(f):,d} points; {RF_IMP_PATHS[plane].name}')
        if RF_TRANSVERSE_MODE == 'iddefix_fit':
            print(f'  Provisional fit. Signed complex relative residual: {records["relative_complex_residual"]:.1%}')

    if RF_FREQ[-1] < 49.999e9:
        raise ValueError('The IDDEFIX HOM-only input must span the full 0–50 GHz budget.')
    if np.min(RF_Z_SINGLE.real) < -1e-7:
        raise ValueError('HOM-only RF source contains unexpected negative Re Z; inspect IDDEFIX export.')

    for pl,(f,z) in STRIPLINE_DATA.items():
        print(f'{pl:4s}: {len(f):>7,d} points, {f[0]/1e9:.4g}–{f[-1]/1e9:.4g} GHz; {DAMPER_PATHS[pl]}')
    print(f'RF HOM-only: {len(RF_FREQ):,d} points; max Re Z = {RF_Z_SINGLE.real.max():.5g} Ω/cavity')

    # Optional source wake: provided as time[ns], Wlong[V/pC], Wx/Wy[V/(pC mm)],
    # with possible additional quadrupole columns. It is NOT substituted into the
    # 0.4 mm budget until its CST bunch sigma and normalization are independently known.
    try:
        DAMPER_ORIGINAL_WAKE_PATH = find_unique_source(DAMPER_WAKE_FILENAME)
        DAMPER_ORIGINAL_WAKE = np.loadtxt(DAMPER_ORIGINAL_WAKE_PATH,comments='#',skiprows=1,ndmin=2)
        if DAMPER_ORIGINAL_WAKE.shape[1] not in (4,6):
            warnings.warn('W_stripkickers26 has unexpected column count; not used in budget')
            DAMPER_ORIGINAL_WAKE = None
        else:
            print('Reference collider wake loaded (diagnostics only):', DAMPER_ORIGINAL_WAKE_PATH,
                  'shape=',DAMPER_ORIGINAL_WAKE.shape)
    except (FileNotFoundError, ValueError) as exc:
        DAMPER_ORIGINAL_WAKE_PATH = None
        DAMPER_ORIGINAL_WAKE = None
        warnings.warn(f'Optional source wake not loaded: {exc}')

In [ ]:
if RUN_MODE == 'native_pywit':
    heb_module = importlib.import_module(HEBModel.__module__)
    old_rf_paths = heb_module.SOURCE_PATHS['RF'].copy()
    try:
        heb_module.SOURCE_PATHS['RF'].update({pl: path.relative_to(DATA).as_posix()
                                            for pl, path in RF_IMP_PATHS.items()})
        native_model = HEBModel(
            energy=ENERGY_EV,
            optics_filename=OPTICS_TFS,
            data_dir=DATA,
            n_rf=112,
            rf_name_hint='ac800',
            sigma_cu=5.8e7,
            sigma_neg=1e6,
            neg_t_m=150e-9,
            roughness_rq_m=0.3e-6,
            gold_reference_q=-2.1043,
            beam_pipe_radius_m=0.030,
            rw_fmax_hz=10e12,
            rw_nfreq=2801,
            kicker_length_m=L_EXTRACTION_M,
            kicker_reference_length_m=COLLIDER_EXTRACTION_REFERENCE_M,
            kicker_count=N_EXTRACTION_MODULES,
            kicker_s_positions_m=EXTRACTION_MODULE_S_POSITIONS_M,
            kicker_archive=KICKER_ARCHIVE,
            wigglers=10,
            wiggler_each_length_m=6.03,
            wiggler_beta_m=4.0,
            wiggler_cache=WIGGLER_CACHE,
            wiggler_iw2d_executable=FLATCHAMBER_EXE,
            n_bellows=10000,
            bellows_length_m=0.15,
        )
    finally:
        heb_module.SOURCE_PATHS['RF'].update(old_rf_paths)

    assert len(native_model.elements) == 6, 'Expected six base PyWIT elements in V26.2'
    print('Base native PyWIT elements:', [el.name for el in native_model.elements])
    rf_provenance = native_model.source_provenance.query("family == 'RF'").set_index('plane')
    for plane, path in RF_IMP_PATHS.items():
        assert Path(rf_provenance.loc[plane, 'path']).resolve() == path.resolve()
        print(f'RF {plane} loaded by PyWIT: {path.name}; ring multiplier {rf_provenance.loc[plane, "multiplier"]:.9g}')
    np.testing.assert_allclose(native_model.rw_length_m, rw_length_m, rtol=0, atol=1e-6)
    np.testing.assert_allclose(native_model.kicker_factor, EXTRACTION_FACTOR, rtol=0, atol=1e-12)

In [ ]:
if RUN_MODE == 'native_pywit':
    if DAMPER_S_POSITIONS_M is None:
        damper_locations = pd.DataFrame({
            'unit':np.arange(1,N_HEB_DAMPERS+1),
            's_m':np.full(N_HEB_DAMPERS,np.nan),
            'beta_x_m':np.full(N_HEB_DAMPERS,native_model.beta_x_avg),
            'beta_y_m':np.full(N_HEB_DAMPERS,native_model.beta_y_avg),
            'status':['unknown; ring-average beta assumption']*N_HEB_DAMPERS,
        })
    else:
        ss=np.asarray(DAMPER_S_POSITIONS_M,float)
        if ss.shape!=(N_HEB_DAMPERS,) or not np.all(np.isfinite(ss)):
            raise ValueError('DAMPER_S_POSITIONS_M must list one finite position per HEB damper')
        optics = native_model.optics_table.sort_values('s_m').drop_duplicates('s_m')
        sopt = optics.s_m.to_numpy(float)
        if ss.min()<sopt.min() or ss.max()>sopt.max():
            raise ValueError('Damper position is outside available V26.2 TFS coordinates')
        damper_locations=pd.DataFrame({
            'unit':np.arange(1,N_HEB_DAMPERS+1),
            's_m':ss,
            'beta_x_m':np.interp(ss,sopt,optics.beta_x_m),
            'beta_y_m':np.interp(ss,sopt,optics.beta_y_m),
            'status':['provisional position; local V26.2 beta']*N_HEB_DAMPERS,
        })

    DAMPER_WEIGHTS = {
        'long': N_HEB_DAMPERS/N_COLLIDER_STRIPLINES,
        'x':float(damper_locations.beta_x_m.sum()/native_model.beta_x_avg/N_COLLIDER_STRIPLINES),
        'y':float(damper_locations.beta_y_m.sum()/native_model.beta_y_avg/N_COLLIDER_STRIPLINES),
    }


    def interpolate_source(freq_src, z_src, freq_dest):
        # No extrapolation outside the source frequency support.
        fq=np.atleast_1d(np.asarray(freq_dest,float))
        return (np.interp(fq,freq_src,z_src.real,left=0.,right=0.)
                +1j*np.interp(fq,freq_src,z_src.imag,left=0.,right=0.))


    def damper_impedance(plane, frequency_hz):
        freq_src,z_src=STRIPLINE_DATA[plane]
        scaled=interpolate_source(freq_src,z_src,frequency_hz)*DAMPER_WEIGHTS[plane]
        return scaled[0] if np.ndim(frequency_hz)==0 else scaled

    # Both the six-component HEBModel and the seventh element use the reference
    # betas and an effective PyWIT length of 1. The impedance arrays are already
    # integrated and (for x/y) HEB beta-weighted.
    damper_components=[]
    for pl,component_plane,exps in [('long','z',(0,0)),('x','x',(1,0)),('y','y',(0,1))]:
        def zfun(frequency, which=pl):
            return damper_impedance(which,frequency)
        damper_components.append(Component(
            impedance=zfun,plane=component_plane,
            source_exponents=exps,test_exponents=(0,0),
            name=f'Damper_{pl}',
        ))

    damper_element=Element(
        length=1.0,beta_x=native_model.beta_x_avg,beta_y=native_model.beta_y_avg,
        components=damper_components,name='Damper',
        description=f'Provisional HEB {N_HEB_DAMPERS} stripline damper(s); collider family / {N_COLLIDER_STRIPLINES}',
    )
    combined_pywit=Model(
        elements=list(native_model.elements)+[damper_element],
        lumped_betas=(native_model.beta_x_avg,native_model.beta_y_avg),
    )

    SHORTHAND={'long':'z0000','x':'x1000','y':'y0100'}
    print('Combined native PyWIT:',[el.name for el in combined_pywit.elements])
    print('Damper family weights: long, x, y =',DAMPER_WEIGHTS)
    display(damper_locations)

    def impedance_budget(plane, freq_hz):
        f=np.asarray(freq_hz,float)
        if np.any(np.diff(f)<=0):
            raise ValueError('Frequency axis must be strictly increasing')
        components={family:np.asarray(native_model.component_impedance(family,plane,f),complex)
                    for family in NATIVE_FAMILIES}
        components['Damper']=np.asarray(damper_impedance(plane,f),complex)
        total=np.asarray(combined_pywit.total.get_component(SHORTHAND[plane]).impedance(f),complex)
        summed=sum(components.values())
        np.testing.assert_allclose(total,summed,rtol=1e-8,atol=1e-7,
            err_msg=f'PyWIT seven-component total inconsistent in {plane}')
        return pd.DataFrame({'frequency_GHz':f/1e9,
            **{f'{kind}_{family}':mapper(z) for family,z in {**components,'Total':total}.items()
               for kind,mapper in [('Re',np.real),('Im',np.imag)]}})

    def frequency_union(*grids):
        # Merge equivalent samples after the GHz/Hz conversion in the source loader.
        return np.unique(np.rint(np.concatenate(grids))).astype(float)


    budget = {'long': impedance_budget('long', F_BUDGET)}
    for plane, (f_rf, _) in RF_TRANSVERSE_DATA.items():
        budget[plane] = impedance_budget(plane, frequency_union(F_BUDGET, f_rf))
    # The adaptive IDDEFIX grid already contains 10 MHz samples, but take a union
    # for safety if that changes in a future export.
    # Round to 1 Hz before merging: source GHz->Hz floating errors can otherwise
    # create almost-duplicate 10MHz nodes that collapse on file export.
    F_ADAPTIVE=frequency_union(F_BUDGET, RF_FREQ)
    assert F_ADAPTIVE[-1] <= 50e9*1.00000001
    budget_adaptive_long=impedance_budget('long',F_ADAPTIVE)

    # HOM-only RF loss factor: integrate on the ADAPTIVE grid, not 10 MHz.
    def loss_factor_from_signed_complex(f_hz,z,sigma_m):
        w=np.exp(-(2*np.pi*f_hz*sigma_m/speed_of_light)**2)
        from scipy.integrate import trapezoid
        return 2e-12*trapezoid(np.asarray(z).real*w,f_hz)

    rf_kappa_per_cavity=loss_factor_from_signed_complex(RF_FREQ,RF_Z_SINGLE,4e-3)
    rf_kappa_112=rf_kappa_per_cavity*native_model.n_rf
    print(f'IDDEFIX HOM-only longitudinal RF, sigma=4 mm: {rf_kappa_per_cavity:.6f} V/pC/cavity; '
          f'{rf_kappa_112:.6f} V/pC for {native_model.n_rf} cavities')
    print('0–50 GHz budget points:',len(F_BUDGET),'Adaptive longitudinal points:',len(F_ADAPTIVE))

    for pl,table in budget.items():
        deficit=int(np.sum(table['Re_Total']<table['Re_RW'])) if pl=='long' else None
        print(pl,'signed table shape:',table.shape,
              ('total ReZ below RW at '+str(deficit)+' 10MHz points' if pl=='long' else ''))

In [ ]:
if RUN_MODE == 'native_pywit':
    from scipy.constants import c as c_light

    native_wakes=native_model.bunch_wake_04mm(
        sigma_z_m=WAKE_SIGMA_M,
        fmax_hz=WAKE_FMAX_HZ,
        df_hz=WAKE_DF_HZ,
        s_window_mm=WAKE_WINDOW_MM,
        include_individual=True,
    )
    reference_s=np.asarray(native_wakes['s_mm'],float)

    def ring_rf_wake(plane):
        path = RF_WAKE_PATHS[plane]
        data = np.loadtxt(path, comments='#', ndmin=2)
        if data.shape[1] != 2 or not np.all(np.isfinite(data)):
            raise ValueError(f'Expected finite s [mm] and wake columns in {path}')
        rf_s, rf_w = data.T
        if not np.all(np.diff(rf_s) > 0) or rf_s[0] > reference_s[0] or rf_s[-1] < reference_s[-1]:
            raise ValueError(f'Invalid s grid or insufficient coverage of the HEB wake window in {path}')
        return np.interp(reference_s, rf_s, rf_w) * float(rf_provenance.loc[plane, 'multiplier'])


    # Match the exact PyWIT HEBModel's irfft convention and unit multipliers.
    # Entire damper is already beta-weighted; this calculation is NOT per m.
    n=int(np.rint(WAKE_FMAX_HZ/WAKE_DF_HZ))
    if n<1 or not np.isclose(n*WAKE_DF_HZ,WAKE_FMAX_HZ,rtol=1e-12):
        raise ValueError('WAKE_FMAX_HZ/WAKE_DF_HZ should be an integer')
    f_fft=np.arange(n+1,dtype=float)*WAKE_DF_HZ
    dt=1./(2*n*WAKE_DF_HZ)
    s_full_mm=(np.arange(2*n)-n)*dt*c_light*1e3
    mask=(s_full_mm>=WAKE_WINDOW_MM[0])&(s_full_mm<=WAKE_WINDOW_MM[1])
    np.testing.assert_allclose(s_full_mm[mask],reference_s,rtol=0,atol=3e-9)
    g=np.exp(-0.5*(2*np.pi*f_fft*WAKE_SIGMA_M/c_light)**2)

    def damper_gaussian_wake(plane):
        z=np.asarray(damper_impedance(plane,f_fft),complex)
        spec=z*g if plane=='long' else -1j*z*g
        spec[0]=complex(spec[0].real,0)
        spec[-1]=complex(spec[-1].real,0)
        physical=np.fft.fftshift(np.fft.irfft(spec,n=2*n))/dt
        conv=-1e-12 if plane=='long' else 1e-15
        return physical[mask]*conv

    wakes={'s_mm':reference_s}
    for plane in ('long','x','y'):
        t=native_wakes[plane].copy()
        t['RF'] = ring_rf_wake(plane)
        t['Damper']=damper_gaussian_wake(plane)
        t['Total']=sum(t[family].to_numpy(float) for family in FAMILIES)
        np.testing.assert_allclose(t['Total'],sum(t[fam] for fam in FAMILIES),rtol=1e-12,atol=1e-10)
        wakes[plane]=t
        print(f'{plane}: {len(t):,d} wake points; RF source {RF_WAKE_PATHS[plane].name}')

## 4. Recalculate from the component exports

The published-components route applies the length ratios independently to the RW and extraction spectra and their 0.4 mm wakes. It sums the seven components on the original frequency and distance grids. The other five component curves are taken directly from the verified reference.

In [ ]:
def update_component_table(table, axis, prefixes):
    updated = table.copy()
    grid = table[axis].to_numpy(dtype=float)
    if not np.all(np.isfinite(table.to_numpy(dtype=float))) or np.any(np.diff(grid) <= 0):
        raise ValueError(f'Invalid reference grid or values in {axis}')
    for prefix in prefixes:
        columns = [prefix + family for family in FAMILIES]
        np.testing.assert_allclose(table[columns].sum(axis=1), table[prefix + 'Total'],
                                   rtol=1e-10, atol=1e-7)
        updated[prefix + 'RW'] *= rw_ratio
        updated[prefix + 'Kicker'] *= extraction_ratio
        updated[prefix + 'Total'] = updated[columns].sum(axis=1)
    return updated


reference_wakes = {
    plane: read_reference_csv(f'component_wakes_{stem}_sigma0p4mm.csv')
    for plane, stem in plane_stems.items()
}
if RUN_MODE == 'published_components':
    for key, expected in {
        'RF_transverse': RF_TRANSVERSE_MODE,
        'HEB_damper_count': N_HEB_DAMPERS,
        'assumed_coll_units': N_COLLIDER_STRIPLINES,
        'HEB_damper_positions': DAMPER_S_POSITIONS_M,
        'active_m_per_kicker': ACTIVE_LENGTH_PER_DAMPER_M,
        'wake_sigma_m': WAKE_SIGMA_M,
        'wake_df_Hz_for_nonRF_and_Damper': WAKE_DF_HZ,
    }.items():
        if reference_manifest[key] != expected:
            raise ValueError(f'The published reference has a different {key}. Use the native PyWIT route.')
    reference_multipliers = reference_provenance.pivot(index='family', columns='plane', values='multiplier')
    np.testing.assert_allclose(reference_multipliers.loc['Kicker'], reference_extraction_m / COLLIDER_EXTRACTION_REFERENCE_M)
    np.testing.assert_allclose(reference_multipliers.loc['RW'], reference_rw_m)
    budget = {
        plane: update_component_table(
            read_reference_csv(f'component_budget_{stem}.csv'), 'frequency_GHz', ('Re_', 'Im_'),
        )
        for plane, stem in plane_stems.items()
    }
    budget_adaptive_long = update_component_table(
        read_reference_csv('adaptive_long_budget.csv'), 'frequency_GHz', ('Re_', 'Im_'),
    )
    wakes = {plane: update_component_table(table, 's_mm', ('',)) for plane, table in reference_wakes.items()}
    wakes['s_mm'] = wakes['long'].s_mm.to_numpy(dtype=float)
    np.testing.assert_allclose(wakes['s_mm'], reference_wakes['x'].s_mm, rtol=0, atol=1e-10)
    np.testing.assert_allclose(wakes['s_mm'], reference_wakes['y'].s_mm, rtol=0, atol=1e-10)
    n_fft = int(round(WAKE_FMAX_HZ / WAKE_DF_HZ))
    fft_s_mm = (np.arange(2 * n_fft) - n_fft) * speed_of_light * 1e3 / (2 * n_fft * WAKE_DF_HZ)
    fft_mask = (fft_s_mm >= WAKE_WINDOW_MM[0]) & (fft_s_mm <= WAKE_WINDOW_MM[1])
    np.testing.assert_allclose(wakes['s_mm'], fft_s_mm[fft_mask], rtol=0, atol=1e-10)
    np.testing.assert_allclose(budget['long'].frequency_GHz, F_BUDGET / 1e9, rtol=0, atol=1e-10)

for plane in plane_stems:
    frame = wakes[plane]
    np.testing.assert_allclose(frame.s_mm, wakes['s_mm'], rtol=0, atol=1e-10)
    np.testing.assert_allclose(frame[list(FAMILIES)].sum(axis=1), frame.Total, rtol=1e-12, atol=1e-10)
    for part in ('Re_', 'Im_'):
        fields = [part + family for family in FAMILIES]
        np.testing.assert_allclose(budget[plane][fields].sum(axis=1), budget[plane][part + 'Total'],
                                   rtol=1e-10, atol=1e-7)
    if RUN_MODE == 'published_components':
        old_wake = reference_wakes[plane]
        for family in ('Wiggler', 'RF', 'BPM', 'Bellows', 'Damper'):
            np.testing.assert_array_equal(frame[family], old_wake[family])
        expected_delta = (extraction_ratio - 1) * old_wake.Kicker + (rw_ratio - 1) * old_wake.RW
        np.testing.assert_allclose(frame.Total - old_wake.Total, expected_delta, rtol=1e-9, atol=1e-9)

wake_delta = pd.DataFrame([
    {
        'Plane': plane_stems[plane],
        'Wake units': 'V/pC' if plane == 'long' else 'V/(pC mm)',
        'Peak |previous total|': np.max(np.abs(reference_wakes[plane].Total)),
        'Peak |new total|': np.max(np.abs(wakes[plane].Total)),
        'Max |new minus previous|': np.max(np.abs(
            wakes[plane].Total - np.interp(wakes[plane].s_mm, reference_wakes[plane].s_mm,
                                           reference_wakes[plane].Total))),
    }
    for plane in plane_stems
]).set_index('Plane')
display(wake_delta)

## 5. Inspect the signed impedance and the tracking wake

In [ ]:
colors = {'RW': '#217188', 'Wiggler': '#805fb7', 'Kicker': '#2ca02c',
          'RF': '#d62728', 'BPM': '#e5a12c', 'Bellows': '#8c564b', 'Damper': '#1ca6a3'}
labels = {'Kicker': 'Extraction kickers', 'Damper': 'Damper striplines'}
plane_labels = {'long': 'Longitudinal', 'x': 'Horizontal dipole', 'y': 'Vertical dipole'}
if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for plane, stem in plane_stems.items():
    table = budget_adaptive_long if plane == 'long' else budget[plane]
    figure, axes = plt.subplots(2, 1, figsize=(11, 7), layout='constrained', sharex=True)
    for ax, part in zip(axes, ('Re', 'Im')):
        for family in FAMILIES:
            ax.plot(table.frequency_GHz, table[f'{part}_{family}'], lw=1.1,
                    color=colors[family], label=labels.get(family, family))
        ax.plot(table.frequency_GHz, table[f'{part}_Total'], color='black', lw=1.7, label='Total')
        unit = 'Ω' if plane == 'long' else 'Ω/m'
        ax.set(yscale='symlog', ylabel=f'{part}(Z) [{unit}]', xlim=(0, 50))
        ax.grid(alpha=.23)
    axes[0].legend(ncol=4, fontsize=8, frameon=False)
    axes[-1].set_xlabel('Frequency [GHz]')
    figure.suptitle(f'{plane_labels[plane]} impedance · {N_EXTRACTION_MODULES} extraction modules')
    if SAVE_FIGURES:
        figure.savefig(FIGURES_DIR / f'impedance_{stem}.png', dpi=160)
    plt.show()
    plt.close(figure)

figure, axes = plt.subplots(3, 2, figsize=(13, 9), layout='constrained', sharex=True)
for row, plane in enumerate(plane_stems):
    current = wakes[plane]
    previous = reference_wakes[plane]
    unit = 'V/pC' if plane == 'long' else 'V/(pC mm)'
    axes[row, 0].plot(previous.s_mm, previous.Total, '--', color='#536271', label='V26.2.1 reference')
    axes[row, 0].plot(current.s_mm, current.Total, color='#bb4c21', label=f'{N_EXTRACTION_MODULES} modules')
    axes[row, 0].set_title(f'{plane_labels[plane]} total')
    for family in ('Kicker', 'RW', 'Total'):
        baseline = np.interp(current.s_mm, previous.s_mm, previous[family])
        axes[row, 1].plot(current.s_mm, current[family] - baseline,
                          color=colors.get(family, 'black'), label=labels.get(family, family))
    axes[row, 1].set_title(f'{plane_labels[plane]} change')
    for ax in axes[row]:
        ax.set_ylabel(unit)
        ax.grid(alpha=.23)
for ax in axes[-1]:
    ax.set_xlabel('Trailing distance s [mm]')
for ax in axes[0]:
    ax.legend(frameon=False)
figure.suptitle('FCC-ee HEB Gaussian bunch wake · σz = 0.4 mm')
if SAVE_FIGURES:
    figure.savefig(FIGURES_DIR / 'wake_comparison.png', dpi=160)
plt.show()
plt.close(figure)

## 6. Export and verify the PyHEADTAIL ECSV and signed budget

In [ ]:
def project_path(path):
    path = Path(path).resolve()
    return path.relative_to(PROJECT_ROOT).as_posix() if path.is_relative_to(PROJECT_ROOT) else str(path)


def write_z(path, table):
    np.savetxt(path, table[['frequency_GHz', 'Re_Total', 'Im_Total']].to_numpy(), fmt='%.14e',
               header='FCC-ee HEB signed ring total\nfrequency_GHz ReZ ImZ')


def write_w(path, table, plane):
    units = 'V/pC' if plane == 'long' else 'V/(pC mm)'
    np.savetxt(path, table[['s_mm', 'Total']].to_numpy(), fmt='%.14e',
               header=f'FCC-ee HEB ring wake, 0.4 mm Gaussian, {units}\ns_mm wake')


if RUN_MODE == 'published_components':
    inventory = reference_inventory.reset_index().copy()
    inventory.loc[inventory.component == 'Kicker', ['count', 'physical_length_m']] = [
        N_EXTRACTION_MODULES, L_EXTRACTION_M,
    ]
    inventory.loc[inventory.component == 'Kicker', 'location_status'] = (
        f'{N_EXTRACTION_SECTIONS} Point B section(s); ring-average beta pending module positions'
    )
    inventory.loc[inventory.component == 'RW', 'physical_length_m'] = rw_length_m
    provenance = reference_provenance.copy()
    for family, scale in (('RW', rw_ratio), ('Kicker', extraction_ratio)):
        rows = provenance.family == family
        provenance.loc[rows, 'multiplier'] *= scale
        provenance.loc[rows, 'normalization'] = (
            f'V26.2.1 published component scaled by {scale:.16g}; '
            + ('circular pipe after subtracting extraction length'
               if family == 'RW' else 'length-scaled collider reference; ring-average beta')
        )
    damper_locations = reference_damper_locations
    rf_fields = {key: value for key, value in reference_manifest.items()
                 if key.startswith(('RF_', 'stripline_', 'assumed_coll_',
                                    'assumption_Coll_', 'HEB_damper_', 'damper_weights',
                                    'active_m_per_kicker', 'warning_normalization'))}
else:
    inventory = pd.concat([native_model.inventory, pd.DataFrame([{
        'component': 'Damper', 'count': N_HEB_DAMPERS,
        'physical_length_m': N_HEB_DAMPERS * ACTIVE_LENGTH_PER_DAMPER_M,
        'location_status': 'provisional TFS positions' if DAMPER_S_POSITIONS_M is not None else 'ring-average beta',
    }])], ignore_index=True)
    provenance = native_model.source_provenance.copy()
    for plane, path in RF_WAKE_PATHS.items():
        provenance.loc[(provenance.family == 'RF') & (provenance.plane == plane), 'wake_path'] = project_path(path)
    provenance = pd.concat([provenance, pd.DataFrame([{
        'family': 'Damper', 'plane': plane, 'path': project_path(DAMPER_PATHS[plane]),
        'normalization': f'Collider system/{N_COLLIDER_STRIPLINES} weighted by HEB module count and beta',
        'multiplier': DAMPER_WEIGHTS[plane], 'input_fmax_ghz': STRIPLINE_DATA[plane][0][-1] / 1e9,
    } for plane in plane_stems])], ignore_index=True)
    rf_fields = {
        'RF_long_source': project_path(RF_IMP_PATH),
        'RF_long_bunch_wake_source': project_path(RF_WAKE_PATH),
        'RF_transverse': RF_TRANSVERSE_MODE,
        'RF_transverse_source_files': {p: project_path(RF_IMP_PATHS[p]) for p in ('x', 'y')},
        'RF_transverse_bunch_wakes': {p: project_path(RF_WAKE_PATHS[p]) for p in ('x', 'y')},
        'RF_transverse_manifest': project_path(RF_TRANSVERSE_MANIFEST_PATH),
        'RF_ring_multipliers': {p: float(rf_provenance.loc[p, 'multiplier']) for p in plane_stems},
        'RF_loss_factor_4mm_V_per_pC_112cav': rf_kappa_112,
        'stripline_source_files': {p: project_path(DAMPER_PATHS[p]) for p in plane_stems},
        'assumed_coll_units': N_COLLIDER_STRIPLINES,
        'HEB_damper_count': N_HEB_DAMPERS,
        'HEB_damper_positions': DAMPER_S_POSITIONS_M,
        'active_m_per_kicker': ACTIVE_LENGTH_PER_DAMPER_M,
        'damper_weights': DAMPER_WEIGHTS,
    }

if SAVE_PROJECT_DATA:
    TOTAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    prefix = f'HEB_{CASE_TAG}'
    saved = []
    for plane, stem in plane_stems.items():
        z_path = TOTAL_EXPORT_DIR / f'Z_total_{stem}_{CASE_TAG}.txt'
        w_path = TOTAL_EXPORT_DIR / f'W_total_{stem}_sigma0p4mm_{CASE_TAG}.txt'
        z_components = TOTAL_EXPORT_DIR / f'{prefix}_component_budget_{stem}.csv'
        w_components = TOTAL_EXPORT_DIR / f'{prefix}_component_wakes_{stem}_sigma0p4mm.csv'
        write_z(z_path, budget[plane])
        write_w(w_path, wakes[plane], plane)
        budget[plane].to_csv(z_components, index=False)
        wakes[plane].to_csv(w_components, index=False)
        saved.extend((z_path, w_path, z_components, w_components))

    adaptive_z = TOTAL_EXPORT_DIR / f'Z_total_long_{CASE_TAG}_ADAPTIVE_0to50GHz.txt'
    adaptive_budget = TOTAL_EXPORT_DIR / f'{prefix}_adaptive_long_budget.csv'
    write_z(adaptive_z, budget_adaptive_long)
    budget_adaptive_long.to_csv(adaptive_budget, index=False)
    saved.extend((adaptive_z, adaptive_budget))
    for table, name in ((inventory, 'inventory'), (provenance, 'source_provenance'),
                        (damper_locations, 'Damper_locations')):
        path = TOTAL_EXPORT_DIR / f'{prefix}_{name}.csv'
        table.to_csv(path, index=False)
        saved.append(path)

    headtail = TOTAL_EXPORT_DIR / f'{prefix}_sigma0p4mm_PyHEADTAIL.ecsv'
    time_ns = wakes['s_mm'] * 1e6 / speed_of_light
    tracking = np.column_stack([time_ns, *(wakes[p].Total for p in plane_stems)])
    columns = ('time', 'longitudinal', 'dipole_x', 'dipole_y')
    with headtail.open('w', encoding='utf-8') as handle:
        handle.write('# %ECSV 1.0\n# ---\n# datatype:\n')
        for name in columns:
            handle.write(f'# - {{name: {name}, datatype: float64}}\n')
        handle.write('# meta: !!omap\n')
        handle.write(f'# - {{description: "FCC-ee HEB V26.2 LCC, {N_EXTRACTION_MODULES} extraction modules, 0.4 mm wake"}}\n')
        handle.write('# schema: astropy-2.0\n')
        handle.write(' '.join(columns) + '\n')
        np.savetxt(handle, tracking, fmt='%.14e')
    readback = pd.read_csv(headtail, sep=r'\s+', comment='#', float_precision='round_trip')
    if tuple(readback.columns) != columns or not np.all(np.diff(readback.time) > 0):
        raise ValueError('Invalid PyHEADTAIL ECSV columns or time grid.')
    np.testing.assert_allclose(readback.to_numpy(), tracking, rtol=8e-15, atol=1e-14)
    saved.append(headtail)

    manifest = {
        **rf_fields,
        'case_tag': CASE_TAG,
        'description': 'FCC-ee HEB V26.2 LCC extraction correction with rough NEG/Cu and IDDEFIX RF',
        'notebook': 'Example/HEB_V26_2_LCC_Extraction_Budget_and_Wake.ipynb',
        'date_utc': datetime.now(timezone.utc).isoformat(),
        'computation': RUN_MODE,
        'reference_case': REFERENCE_TAG,
        'reference_inputs_sha256': {project_path(path): sha256(path.read_bytes()).hexdigest()
                                    for path in reference_inputs},
        'optics': project_path(OPTICS_TFS),
        'optics_sha256': sha256(OPTICS_TFS.read_bytes()).hexdigest(),
        'extraction_sections': N_EXTRACTION_SECTIONS,
        'extraction_modules_per_section': MODULES_PER_EXTRACTION_SECTION,
        'extraction_module_count': N_EXTRACTION_MODULES,
        'extraction_magnetic_length_per_module_m': MAGNETIC_LENGTH_PER_MODULE_M,
        'extraction_total_magnetic_length_m': L_EXTRACTION_M,
        'extraction_collider_reference_length_m': COLLIDER_EXTRACTION_REFERENCE_M,
        'extraction_longitudinal_multiplier': EXTRACTION_FACTOR,
        'extraction_module_s_positions_m': EXTRACTION_MODULE_S_POSITIONS_M,
        'extraction_transverse_weighting': (
            'ring average, provisional' if EXTRACTION_MODULE_S_POSITIONS_M is None else 'local module beta'
        ),
        'rw_circumference_m': circumference_m,
        'rw_bellows_excluded_m': bellows_m,
        'rw_wigglers_excluded_m': wiggler_m,
        'rw_previous_pipe_length_m': reference_rw_m,
        'rw_pipe_length_m': rw_length_m,
        'rw_scale_against_reference': rw_ratio,
        'rw_model': {'substrate': 'Cu', 'NEG_thickness_m': 150e-9, 'roughness_Rq_m': 0.3e-6,
                     'pipe_radius_m': 0.03, 'gold_reference_q': -2.1043},
        'wake_sigma_m': WAKE_SIGMA_M,
        'wake_df_Hz_for_nonRF_and_Damper': WAKE_DF_HZ,
        'wake_fmax_Hz': WAKE_FMAX_HZ,
        'wake_window_mm': list(WAKE_WINDOW_MM),
        'wake_points': len(tracking),
        'frequency_units': 'GHz',
        'impedance_units_long': 'Ohm',
        'impedance_units_dipoles': 'Ohm/m',
        'wake_units_long': 'V/pC',
        'wake_units_dipoles': 'V/pC/mm',
        'ecsv_time_units': 'ns',
        'wake_file': headtail.name,
        'wake_sha256': sha256(headtail.read_bytes()).hexdigest(),
        'wake_comparison': wake_delta.reset_index().to_dict(orient='records'),
        'saved_files': [project_path(path) for path in saved],
    }
    manifest_path = TOTAL_EXPORT_DIR / f'{prefix}_EXPORT_MANIFEST.json'
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    print(f'Exported {len(saved) + 1} files to {TOTAL_EXPORT_DIR}')
    print(f'PyHEADTAIL wake: {headtail.name}')
    print(f'SHA-256: {manifest["wake_sha256"]}')

The export manifest records the design count, effective lengths, beta treatment and wake SHA-256. The adaptive longitudinal file is the appropriate impedance table for narrow HOMs. The transverse model should use local Point B beta functions when the equipment positions become available.